# COLAB

In [4]:
import os, pickle
d = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center"

# 1. ¿El vídeo está completo?
print(os.path.getsize(f"{d}/video.avi") == 9508744968)

# 2. ¿El pkl se abre?
with open(f"{d}/gt_data.pkl", "rb") as f:
    gt = pickle.load(f, encoding="latin1")

True


In [7]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.7 MB/s eta 0:00:00


In [34]:
COLORES = {2: (0, 255, 0), 3: (255, 0, 255), 5: (0, 165, 255), 7: (0, 0, 255)}  # BGR: coche verde, camión rojo

CLIPS_TRACKING = {
    "simple" : {
        "carId" : 72,
        "margen" : 0.5,
        "nota" : "turismo solo, carril izquierdo, sin mas coches"
    },

    "paralelo" : {
        "carId" : 350,
        "margen" : 0.5,
        "nota" : "varios coches en escena, en ambos carriles"
    },

    "adelantamiento" : {
        "carId" : 65,
        "margen" : 0.5,
        "nota" : "varios coches en ambos carriles, coche adelantando camion por izquierda"
    }
}

ruta = "/content/drive/MyDrive/radar-trafico/data/raw/session2_center/video.avi"
mask = cv2.imread(f"{d}/video_mask.png", cv2.IMREAD_GRAYSCALE)


In [1]:
modelo = YOLO("yolo11n.pt")

car = coche_por_id(gt, CLIPS_TRACKING["simple"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])
df = detectar_clip(ruta, tInicio - 1.0, tFinal+ 0.25, mask)

dibujar_df(ruta, tInicio - 1.0, tFinal + 0.25, df)

!ffmpeg -y -loglevel error -i detecciones.mp4 -c:v libx264 detecciones_h264.mp4
# display(Video("detecciones_h264.mp4", embed=True))

NameError: name 'YOLO' is not defined

In [ ]:
def tiempo_entre_lineas(tray, ln4, ln0):

    t4 = t0 = None

    for fila in tray.itertuples():
        x = int((fila.x1 + fila.x2) / 2)

        if t4 is None and fila.y2 >= ln4[x]:
            t4 = fila.t
        if t0 is None and fila.y2 >= ln0[x]:
            t0 = fila.t

    if t4 is None or t0 is None:
        return None

    return (t0 - t4)


In [39]:
tray = df[df["id"] == df["id"].mode()[0]].sort_values("frame")
print("mi radar:", tiempo_entre_lineas(tray), "| GT:", tFinal - tInicio)

mi radar: 1.32000000000005 | GT: 1.3132339999999658


In [ ]:
tiempo = tiempo_entre_lineas(tray)

In [ ]:
carril = next(iter(car["laneIndex"]))
distancias = []

for c in gt["cars"]:
    if c["valid"] and len(c["intersections"]) == 2 and c["intersections"][1]["videoTime"] < 1200:
        dt_gt = c["intersections"][1]["videoTime"] - c["intersections"][0]["videoTime"]
        distancias.append(c["speed"] / 3.6 * dt_gt)

D = np.median(distancias)

In [ ]:
velocidad = D / tiempo * 3.6
print(f"mi radar: {velocidad:.1f} km/h | LiDAR: {car['speed']:.1f} km/h | error: {velocidad - car['speed']:+.1f} km/h")

In [ ]:
def calibrar_distancia(gt, t_max=1200):
    distancias = []
    for c in gt["cars"]:
        if c["valid"] and len(c["intersections"]) == 2 and c["intersections"][1]["videoTime"] < t_max:
            dt = c["intersections"][1]["videoTime"] - c["intersections"][0]["videoTime"]
            distancias.append(c["speed"] / 3.6 * dt)
    return np.median(distancias)

In [ ]:
modelo = YOLO("yolo11n.pt")

car = coche_por_id(gt, CLIPS_TRACKING["paralelo"]["carId"])
tInicio, tFinal = tiempoCoche(car["intersections"])
df = detectar_clip(ruta, tInicio - 5.0, tFinal + 1.5, mask)

dibujar_df(ruta, tInicio - 5.0, tFinal + 1.5, df)

!ffmpeg -y -loglevel error -i detecciones.mp4 -c:v libx264 detecciones_h264.mp4
display(Video("detecciones_h264.mp4", embed=True))

In [ ]:
resultados = []

D = calibrar_distancia(gt)

for coche, tray in df[df["id"] >= 0].groupby("id"):
    tray = tray.sort_values("frame")
    tiempo = tiempo_entre_lineas(tray)

    if tiempo is not None:
        resultados.append({"id" : coche, "tiempo" : tiempo, "velocidad" : D / tiempo * 3.6})

print(pd.DataFrame(resultados))

   id  tiempo  velocidad
0   6    1.14  88.421173
1  30    1.04  96.923209
2  35    1.08  93.333461
3  51    1.36  74.117748
